In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Parameters

In [ ]:
n_input = 20
n_hidden = 256
n_out = 5
batch_size = 100
learning_rate = 0.01
agent_n_epochs = 5000
adversary_n_epochs = 5000

### Data

In [ ]:
# load dataframe from pickle
df1 = pd.read_pickle('checkpoint_004001_data_1.pkl')
df2 = pd.read_pickle('checkpoint_004001_data_2.pkl')
df3 = pd.read_pickle('checkpoint_004001_data_3.pkl')

print(np.shape(df1))
print(np.shape(df2))
print(np.shape(df3))

data = df1.append(df2, ignore_index=True).append(df3, ignore_index=True)
print(np.shape(data))

df1 = df2 = df3 = None

# data = pd.read_pickle('checkpoint_004001_data.pkl')

In [ ]:
df = data[['obs','actions','policy_name']]
df[['obs_0','obs_1','obs_2','obs_3','obs_4','obs_5','obs_6','obs_7','obs_8','obs_9', \
    'obs_10','obs_11','obs_12','obs_13','obs_14','obs_15','obs_16','obs_17','obs_18','obs_19']] = \
    pd.DataFrame(df.obs.tolist(), index= df.index)

## Functions

In [ ]:
def data_prep(policy_df):

    training_data, testing_data = train_test_split(policy_df, test_size=0.2, random_state=42, shuffle=True)

    inputs_df_train = pd.DataFrame(data=training_data[['obs_0','obs_1','obs_2','obs_3','obs_4','obs_5','obs_6','obs_7','obs_8','obs_9', \
                                    'obs_10','obs_11','obs_12','obs_13','obs_14','obs_15','obs_16','obs_17','obs_18','obs_19']])
    inputs_df_test = pd.DataFrame(data=testing_data[['obs_0','obs_1','obs_2','obs_3','obs_4','obs_5','obs_6','obs_7','obs_8','obs_9', \
                                    'obs_10','obs_11','obs_12','obs_13','obs_14','obs_15','obs_16','obs_17','obs_18','obs_19']])   

    inputs_tensor_train = torch.Tensor(inputs_df_train.values)  
    inputs_tensor_test = torch.Tensor(inputs_df_test.values)                     
    
    targets_df_train = pd.DataFrame(data=training_data[['actions']])
    target_tensor_train = torch.tensor(targets_df_train.values, dtype=torch.long)
    target_tensor_train = torch.squeeze(target_tensor_train)
    targets_df_test = pd.DataFrame(data=testing_data[['actions']])
    target_tensor_test = torch.tensor(targets_df_test.values, dtype=torch.long)
    target_tensor_test = torch.squeeze(target_tensor_test)


    print("Input Train Size:", inputs_tensor_train.size())
    print("Input Test Size:", inputs_tensor_test.size())    
    print("Output Train Size:", target_tensor_train.size())
    print("Output Test Size:", target_tensor_test.size())


    return inputs_tensor_train, inputs_tensor_test, target_tensor_train, target_tensor_test

In [ ]:
def policy_model():
    model = nn.Sequential(nn.Linear(n_input, n_hidden, bias=True),
                      nn.Tanh(),
                      nn.Linear(n_hidden, n_hidden, bias=True),
                      nn.Tanh(),
                      nn.Linear(n_hidden, n_out, bias=True))
    print(model)

    loss_function = nn.CrossEntropyLoss()  
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    return model, loss_function, optimizer

In [ ]:
def training_and_testing(model, n_epochs, data_X_train, data_Y_train, data_X_test, data_Y_test, loss_function, optimizer):
    train_losses = []
    train_x_axis = []
    test_losses = []
    test_x_axis = []
    for epoch in range(n_epochs):
        
        if epoch% 5 == 0 and epoch != 0:
            test_pred_y = model(data_X_test)
            test_loss = loss_function(test_pred_y, data_Y_test)
            test_losses.append(test_loss.item())
            test_x_axis.append(epoch)
        else:
            train_pred_y = model(data_X_train)
            train_loss = loss_function(train_pred_y, data_Y_train)
            train_losses.append(train_loss.item())
            train_x_axis.append(epoch)

            model.zero_grad()
            train_loss.backward()

            optimizer.step()


    plt.plot(test_x_axis, test_losses, label = 'Testing')
    plt.plot(train_x_axis, train_losses, label = 'Training')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend()
    plt.title("Learning rate %f"%(learning_rate))
    plt.show()

## AGENT

In [ ]:
agent_df = df[df['policy_name']=='policy_agent']
agent_df = agent_df.drop(columns=['obs', 'policy_name'])

print(np.shape(agent_df))

In [ ]:
agent_train_X, agent_test_X, agent_train_Y, agent_test_Y = data_prep(agent_df)

In [ ]:
agent_model, agent_loss_function, agent_optimizer = policy_model()

In [ ]:
training_and_testing(agent_model, agent_n_epochs, agent_train_X, agent_train_Y, agent_test_X, agent_test_Y, agent_loss_function, agent_optimizer)

## ADVERSARY

In [ ]:
adversary_df = df[df['policy_name']=='policy_adversary']
adversary_df = adversary_df.drop(columns=['obs', 'policy_name'])

print(np.shape(adversary_df))

In [ ]:
adversary_train_X, adversary_test_X, adversary_train_Y, adversary_test_Y = data_prep(adversary_df)

In [ ]:
adversary_model, adversary_loss_function, adversary_optimizer = policy_model()

In [ ]:
training_and_testing(adversary_model, adversary_n_epochs, adversary_train_X, adversary_train_Y, adversary_test_X, adversary_test_Y, adversary_loss_function, adversary_optimizer)